## **Projeto:** Merca Data Platform

##**Squad:** 2 | Camada Bronze
### Objetivo
Ingerir os dados brutos do catálogo de produtos na camada Bronze do Delta Lake.
Nenhuma transformação é aplicada ao conteúdo — apenas colunas de auditoria e particionamento temporal são adicionados para otimizar consultas futuras.
### Origem e Destino
| **Origem** | `real-time-data/<snapshot>/ecommerce_produtos.parquet` (ADLS) |
| **Destino** | `squad2/bronze/ecommerce_produtos` (Delta Lake) |
| **Checkpoint** | `squad2/control/ecommerce_produtos/control_file.json` |
| **Modo de escrita** | `append` incremental por snapshot |
| **Particionamento** | `ingestion_year / ingestion_month / ingestion_day / ingestion_hour` |
### Colunas de Auditoria Adicionadas
| Coluna | Descrição |
| `bronze_source_file` | Caminho completo do arquivo Parquet de origem |
| `bronze_ingested_at` | Timestamp de ingestão na Bronze |
| `_source` | Fonte dos dados (`real-time-data`) |
| `_camada` | Camada atual (`bronze`) |
| `ingestion_year` | Ano de ingestão (usado como partição) |
| `ingestion_month` | Mês de ingestão (usado como partição) |
| `ingestion_day` | Dia de ingestão (usado como partição) |
| `ingestion_hour` | Hora de ingestão (usado como partição) |
### Dependências
| Notebook | Motivo |
| `feat_squad2_99_helpers` | Conexão ADLS, leitura Parquet, escrita Delta, checkpoint e log |

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
import logging
from pyspark.sql.functions import lit, current_timestamp, year, month, dayofmonth, hour

logging.getLogger("azure").setLevel(logging.WARNING)

TABELA = "ecommerce_produtos"
CAMADA = "bronze"

inicio = log_inicio(f"feat_squad2_{CAMADA}_{TABELA}")
log.info(f"Tabela : {TABELA}")
log.info(f"Camada : {CAMADA}")
log.info(f"Path   : {get_delta_path(CAMADA, TABELA)}")

In [0]:
def processar_snapshot(snapshot_id: str) -> bool:
    """Processa um único snapshot da Bronze de produtos."""
    try:
        df = ler_parquet(snapshot_id, TABELA)

        df_bronze = df \
            .withColumn("_snapshot_id",    lit(snapshot_id)) \
            .withColumn("_ingested_at",    current_timestamp()) \
            .withColumn("_source",         lit("real-time-data")) \
            .withColumn("_camada",         lit(CAMADA)) \
            .withColumn("ingestion_year",  year(current_timestamp()).cast("string")) \
            .withColumn("ingestion_month", month(current_timestamp()).cast("string")) \
            .withColumn("ingestion_day",   dayofmonth(current_timestamp()).cast("string")) \
            .withColumn("ingestion_hour",  hour(current_timestamp()).cast("string"))

        sucesso = gravar_delta(df_bronze, CAMADA, TABELA)

        if sucesso:
            log.info(f" {snapshot_id} → {df_bronze.count()} linhas gravadas.")

        return sucesso

    except Exception as e:
        log.error(f" Erro ao processar {snapshot_id}: {str(e)}")
        return False

### Validação Pontual
Execute esta célula isoladamente para verificar o estado atual sem iniciar o loop contínuo.

In [0]:
try:
    snapshots   = sorted(listar_snapshots())
    processados = ler_checkpoint(CAMADA, TABELA)
    novos       = [s for s in snapshots if s not in processados]

    log.info(f"Snapshots disponíveis : {len(snapshots)}")
    log.info(f"Já processados        : {len(processados)}")
    log.info(f"Novos para processar  : {len(novos)}")
    log.info(f"Status atual          : {ler_status_checkpoint(CAMADA, TABELA)}")

    if not novos:
        log.info("Bronze produtos em dia!")
    else:
        for s in novos:
            print(f"  Processados {s}")

except Exception as e:
    log.error(f"Erro na validação: {str(e)}")
    raise

### Polling Loop
Inicia o monitoramento contínuo da Bronze.
**Bronze não tem dependência de camada anterior.**
Para encerrar, interrompa a execução manualmente.

In [0]:
snapshots   = sorted(listar_snapshots())
processados = ler_checkpoint(CAMADA, TABELA)
novos       = [s for s in snapshots if s not in processados]

if not novos:
    log.info("Bronze " + TABELA + " em dia - nenhum snapshot novo.")
else:
    log.info(str(len(novos)) + " snapshot(s) novo(s) encontrado(s).")
    salvar_checkpoint(CAMADA, TABELA, processados, status="PROCESSANDO")

    for snapshot_id in novos:
        log.info("Processando: " + snapshot_id)
        sucesso = processar_snapshot(snapshot_id)
        if sucesso:
            processados.add(snapshot_id)
            log.info("OK: " + snapshot_id)
        else:
            log.warning("FALHOU: " + snapshot_id + " - sera retentado no proximo ciclo.")

    salvar_checkpoint(CAMADA, TABELA, processados, status="CONCLUIDO")
    log.info("Bronze " + TABELA + " concluida.")

log_fim("feat_squad2_" + CAMADA + "_" + TABELA, inicio)